# How to use this Notebook

1. Please ensure all your pdfs are zipped in one file.
2. Click Runtime, Click change runtime type and select A100. (If you don't have A100 access get Colab pro using your academic email https://blog.google/products-and-platforms/products/education/colab-higher-education/)
3. Please ensure your hugging face token (HF_TOKEN) is includedd in secrets in your Colab Notebook.

If you don't have a hugging face token, you can get one for free. Create an hugging face account and follow steps given on:

https://huggingface.co/docs/hub/en/security-tokens



## Install depencies using cell below


In [1]:
!pip install -q transformers accelerate qwen-vl-utils pypdfium2 pillow pandas

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.4/68.4 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.7/3.7 MB 91.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.4/35.4 MB 72.0 MB/s eta 0:00:00


## Import required libraries using cell below


In [11]:
import os
import json
import re
import zipfile
import pandas as pd
import pypdfium2 as pdfium
from google.colab import files
from qwen_vl_utils import process_vision_info

## Run cell below to and select your zip files containing pdf
After running the cell, you will get an option to choose your zip file containing a pdfs
This cell unzips your zip files and extracts pdfs. It will return number of pdfs found

In [12]:
uploaded = files.upload()
zip_path = next(iter(uploaded.keys()))

extract_dir = "/content/input_pdfs"
os.makedirs(extract_dir, exist_ok=True)

with zipfile.ZipFile(zip_path, "r") as z:
    z.extractall(extract_dir)

pdf_paths = []
for root, dirs, filenames in os.walk(extract_dir):
    for filename in filenames:
        if filename.lower().endswith(".pdf"):
            pdf_paths.append(os.path.join(root, filename))

pdf_paths = sorted(pdf_paths)

print(f"Found {len(pdf_paths)} PDFs")

Saving Field_Notes.zip to Field_Notes.zip
Found 8 PDFs


## Run this cell to load Qwen Model
Please ensure your hugging face token (HF_TOKEN) is includedd in secrets in your Colab Notebook.

If you don't have a hugging face token, you can get one for free. Create an hugging face account and follow steps given on:

https://huggingface.co/docs/hub/en/security-tokens

In [4]:
import torch
from transformers import Qwen2_5_VLForConditionalGeneration, AutoProcessor

model_id = "Qwen/Qwen2.5-VL-32B-Instruct"

model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    model_id,
    torch_dtype=torch.bfloat16,
    device_map="auto"
)

processor = AutoProcessor.from_pretrained(model_id)

config.json:   0%|          | 0.00/1.22k [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/93.1k [00:00<?, ?B/s]

Fetching 18 files:   0%|          | 0/18 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/1161 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/216 [00:00<?, ?B/s]

...

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/605 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/613 [00:00<?, ?B/s]

## Run the cell below to intialize all required functions

In [13]:
def pdf_page_to_image(pdf_path, page_index=1, scale=3):
    pdf = pdfium.PdfDocument(pdf_path)

    if len(pdf) <= page_index:
        pdf.close()
        raise ValueError(f"{pdf_path} does not have page {page_index + 1}")

    page = pdf[page_index]
    pil_image = page.render(scale=scale).to_pil().convert("RGB")

    image_path = f"/content/page_{os.path.basename(pdf_path)}.png"
    image_path = image_path.replace(".pdf.png", "_page2.png")
    pil_image.save(image_path)

    pdf.close()
    return image_path


def extract_json(text):
    text = text.strip()
    text = re.sub(r"^```json\s*", "", text)
    text = re.sub(r"^```\s*", "", text)
    text = re.sub(r"\s*```$", "", text)
    return json.loads(text)


def run_qwen_on_image(image_path):
    prompt = """
You are extracting structured data from a water sampling form.

Look only at the table on the page.

Return JSON only with this exact schema:
{
  "date": "YYYY-MM-DD",
  "rows": [
    {
      "site": "WS-1",
      "algae_present": true,
      "evidence_text": "handwritten text that supports the decision"
    }
  ]
}

Rules:
- Include every site row visible in the table.
- algae_present should be true if the remarks contain algae or a likely handwritten algae mention.
- Treat ALGAE, Algae, ALGAF, ALGA, ALGE, ALGEA, ALGAE-like, or obvious handwritten/misspelled variants as algae_present=true.
- If there is no algae mention, use false.
- Do not include explanations outside JSON.
"""

    messages = [
        {
            "role": "user",
            "content": [
                {"type": "image", "image": image_path},
                {"type": "text", "text": prompt},
            ],
        }
    ]

    text = processor.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    image_inputs, video_inputs = process_vision_info(messages)

    inputs = processor(
        text=[text],
        images=image_inputs,
        videos=video_inputs,
        padding=True,
        return_tensors="pt",
    ).to(model.device)

    generated_ids = model.generate(
        **inputs,
        max_new_tokens=1024
    )

    generated_ids_trimmed = [
        out_ids[len(in_ids):]
        for in_ids, out_ids in zip(inputs.input_ids, generated_ids)
    ]

    output_text = processor.batch_decode(
        generated_ids_trimmed,
        skip_special_tokens=True,
        clean_up_tokenization_spaces=False
    )[0]

    return output_text

## Run the cell below to start the analysis.
You will see which file is being processed below. You will also get to preview outputs once processing is complete.

In [14]:
all_rows = []
failed_files = []

for i, pdf_path in enumerate(pdf_paths, start=1):
    print(f"\nProcessing {i}/{len(pdf_paths)}: {pdf_path}")

    try:
        image_path = pdf_page_to_image(pdf_path, page_index=1, scale=3)
        output_text = run_qwen_on_image(image_path)
        data = extract_json(output_text)

        for row in data["rows"]:
            all_rows.append({
                "source_file": os.path.basename(pdf_path),
                "date": data.get("date", ""),
                "site": row.get("site", ""),
                "algae_present": row.get("algae_present", ""),
                "evidence_text": row.get("evidence_text", "")
            })

    except Exception as e:
        print(f"FAILED: {pdf_path}")
        print(e)
        failed_files.append({
            "source_file": os.path.basename(pdf_path),
            "error": str(e)
        })

df = pd.DataFrame(all_rows)
df


Processing 1/8: /content/input_pdfs/20220308 Notes.pdf

Processing 2/8: /content/input_pdfs/20220314 Notes.pdf

Processing 3/8: /content/input_pdfs/20220321 Notes.pdf

Processing 4/8: /content/input_pdfs/20220329 Notes.pdf

Processing 5/8: /content/input_pdfs/20220404 Notes.pdf

Processing 6/8: /content/input_pdfs/20220411 Notes.pdf

Processing 7/8: /content/input_pdfs/20220418 Notes.pdf

Processing 8/8: /content/input_pdfs/20220425 Notes.pdf


,source_file,date,site,algae_present,evidence_text
0,20220308 Notes.pdf,2022-03-08,WS-1,False,snow/ice bridges
1,20220308 Notes.pdf,2022-03-08,WS-2,False,
2,20220308 Notes.pdf,2022-03-08,WS-3,False,mostly open w/snow bridges
3,20220308 Notes.pdf,2022-03-08,WS-4,True,Algae
4,20220308 Notes.pdf,2022-03-08,WS-5,False,snow/ice bridges
...,...,...,...,...,...
83,20220425 Notes.pdf,2022-04-25,WS-7,False,No snow
84,20220425 Notes.pdf,2022-04-25,WS-8,False,No snow
85,20220425 Notes.pdf,2022-04-25,WS-9,False,"Trace snow, flotsam, noise"
86,20220425 Notes.pdf,2022-04-25,Hubbard BK,True,Algae


## Run the cell below to download your results

In [15]:
output_csv = "/content/algae_results.csv"
df.to_csv(output_csv, index=False)
files.download(output_csv)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

 If you got an Error mesage for any file, Run cell below to get more details

In [ ]:
failed_df = pd.DataFrame(failed_files)
failed_df